In [1]:
import getpass
import os
from dotenv import load_dotenv
# from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFaceEmbeddings
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_postgres import PGVector
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.tools import tool
from langsmith import uuid7
from pydantic import BaseModel
import json

import tiktoken
from langchain_community.callbacks import get_openai_callback
import sqlalchemy

LANGSMITH_KEY = os.getenv('LANGSMITH_API_KEY')
HUGGINGFACE_KEY = os.getenv('HUGGINGFACE_KEY')
OPENAI_KEY = os.getenv('OPEN_AI_KEY')
CONNECTION_STRING = os.getenv('CONNECTION_KEY')
# GEMINI_KEY = os.getnev('GEMINI_KEY')

if LANGSMITH_KEY is not None and HUGGINGFACE_KEY is not None:
    print('Successfully retrieved keys')

c:\Users\panlw\Desktop\ML-helper-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully retrieved keys


In [2]:
# llm = HuggingFaceEndpoint(
#     repo_id="microsoft/Phi-3-mini-4k-instruct",
#     temperature=0.7,
#     huggingfacehub_api_token= HUGGINGFACE_KEY,
# )

model = ChatOpenAI(
    model="gpt-3.5-turbo", openai_api_key=OPENAI_KEY, temperature=0)

# chat_model = ChatHuggingFace(llm = llm)
#chat_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

if model:
    print('model detected')

model detected


In [43]:
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_KEY)

embeddings_v2 = OpenAIEmbeddings(
    openai_api_key=OPENAI_KEY,
    model="text-embedding-3-small"
)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index = False,  # track index in original document
)


In [44]:
loader = PyPDFLoader(r'documents\AAAMLP.pdf')
docs = loader.load()
docs = text_splitter.split_documents(docs)

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 29 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 36 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)
Ignoring wrong pointing object 40 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 47 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 74 0 (offset 0)
Ignoring wrong pointing object 98 0 (offset 0)
Ignoring wrong pointing object 100 0 (offset 0)
Ignoring wrong pointing object 103 0 (offset 0)
Ignoring wron

In [5]:
# for doc in docs:
#     print(doc)
#     break

In [6]:
# page_lengths = [len(doc.page_content) for doc in docs]
# print(page_lengths)

import matplotlib.pyplot as plt

# lengths = [len(doc.page_content) for doc in docs]
# print(f"Chunks: {len(docs)}")
# print(f"Avg: {sum(lengths)/len(lengths):.0f} chars | Min: {min(lengths)} | Max: {max(lengths)}")
# plt.hist(lengths, bins=30); plt.xlabel("chars"); plt.show()

In [45]:
import psycopg2

def collection_has_data(connection_str, collection_name):
    try:

        conn = psycopg2.connect(connection_str)
        cur = conn.cursor()
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        cur.execute("""
            SELECT COUNT(*) FROM langchain_pg_embedding e
            JOIN langchain_pg_collection c ON e.collection_id = c.uuid
            WHERE c.name = %s
        """, (collection_name,))
        count = cur.fetchone()[0]
        conn.close()
        return count > 0
    except:
        return False

connection_str = os.getenv('CONNECTION_KEY')
collection_name = 'ML knowledge'

In [46]:
conn = psycopg2.connect(CONNECTION_STRING)
cur = conn.cursor()
cur.execute("SELECT * FROM pg_available_extensions WHERE name = 'vector';")
result = cur.fetchone()
print(f"Extension details: {result}")
conn.close()

Extension details: ('vector', '0.8.2', '0.8.2', 'vector data type and ivfflat and hnsw access methods')


In [47]:
if collection_has_data(connection_str, collection_name):
    print(f'collection already exist -connecting to existing data')
    db = PGVector(
    embeddings=embeddings,
    # documents=docs,
    collection_name= collection_name,
    connection =connection_str,
)
    
else:
    print(f'collection empty, embedding and inserting documents')
    db = PGVector.from_documents(
    embedding=embeddings,
    documents=docs,
    collection_name= collection_name,
    connection =CONNECTION_STRING,
)
    print(f"Successfully stored {len(docs)} chunks in PostgreSQL.")


collection empty, embedding and inserting documents
Successfully stored 895 chunks in PostgreSQL.


In [ ]:
#testing second db

if collection_has_data(connection_str, 'ml_docs_v2'):
    print(f'collection already exist -connecting to existing data')
    db_v2 = PGVector(
    embeddings=embeddings,
    # documents=docs,
    collection_name= 'ml_docs_v2',
    connection =connection_str,
)
    
else:
    print(f'collection empty, embedding and inserting documents')
    db_v2 = PGVector.from_documents(
    embedding=embeddings,
    documents=docs,
    collection_name= 'ml_docs_v2',
    connection =CONNECTION_STRING,
)
    print(f"Successfully stored {len(docs)} chunks in PostgreSQL.")

# db_v2 = PGVector(
#     embeddings=embeddings_v2,
#     collection_name="ml_docs_v2",  # separate collection
#     connection=CONNECTION_STRING,
# )

In [ ]:
# ingest data into second db

# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1000,  # chunk size (characters)
#     chunk_overlap=200,  # chunk overlap (characters)
#     add_start_index = False,  # track index in original document
# )
# loader = PyPDFLoader(r'documents\AAAMLP.pdf')
# docs = loader.load()
# docs = text_splitter.split_documents(docs)
# # for doc in docs:
# #     print(doc)
# #     break

#db_v2.add_documents(docs)  # same docs variable from your original ingestion

['09fa940e-5279-49c4-8479-439c228cb813',
 '0dbd692e-3b41-42df-9918-04efb1162a6b',
 'c0d0b897-707d-4117-8195-3bf163eae6ed',
 '16167499-f98a-43e8-981d-08ed9f996309',
 'b17bc2bd-beac-4e6e-8d8e-21dd3806cd6c',
 'f2a5269a-1328-49f2-aab3-807fea502f3f',
 '2716f8dd-d92f-46bc-938c-f592634c1754',
 '123da02e-451a-4c16-b654-f085774b8750',
 '42c83aa7-47ae-4216-838d-af81f3118901',
 '83d0642a-1a73-48aa-814c-2f1517554346',
 'a6e15b79-0328-4e0e-bee7-041c6955f593',
 'dda813c8-ad93-463e-986a-e6f9d6a5f1e8',
 '5d53ffb1-4a6a-45bc-a937-adea96549e7a',
 '066d1a3b-3d02-4366-bd2d-225e0c1a89ad',
 '76cf682f-a172-4e4b-9301-8f6e9b8f903d',
 '8bf2f39c-06f0-4519-8a89-c02290ef6e60',
 '2bbeacd6-73da-4e3c-9f1f-08bd06957d53',
 '52d1fe71-8c20-42ae-9f4a-659a293f9f9b',
 '715660a4-7f63-4dcc-98a9-385622968926',
 '026b57f4-0248-4ec7-91ab-141d86c92844',
 '04391459-80c6-4974-8901-4764fb44cc1e',
 '06afc095-b7c9-4744-a461-2a945ba69fe7',
 'ff8f723d-0488-4249-ac4a-e68ef2da50d7',
 'a9da0fa3-88bc-459f-9183-6e9d815fe92e',
 '8a367f29-eda5-

In [12]:
import time

test_questions = [
'what is cross-validation',
'what is overfitting',
'what is stratified k-fold',
'what is supervised learning',
'what is k-fold cross-validation'
]

times = []
for q in test_questions:
    embedded_q = embeddings.embed_query(q)
    start = time.time()
    raw_results = db.similarity_search_with_score_by_vector(embedded_q)
    elapsed = (time.time() - start) * 1000 #ms
    times.append(elapsed)
    print(f"{q[:40]:40} | {elapsed: .2f}ms")

print(f'Average retrieval time: {sum(times)/len(times):.2f}ms')


what is cross-validation                 |  96.48ms
what is overfitting                      |  69.25ms
what is stratified k-fold                |  53.51ms
what is supervised learning              |  62.50ms
what is k-fold cross-validation          |  48.30ms
Average retrieval time: 66.01ms


In [50]:
#HNSW
engine = sqlalchemy.create_engine(CONNECTION_STRING)
with engine.connect() as connec:
    result = connec.execute(sqlalchemy.text(
        "SELECT extversion FROM pg_extension WHERE extname = 'vector'"
    ))
    for row in result:
        print(f"pgvector version: {row[0]}")
    

pgvector version: 0.8.2


In [51]:
with engine.connect() as conn:
    conn.execute(sqlalchemy.text(
    """
    ALTER TABLE  langchain_pg_embedding
    ALTER COLUMN embedding TYPE vector(1536)
    """

    ))
    conn.commit()
    print("column dimension set to 1536")

column dimension set to 1536


In [52]:
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text(
        """
        SELECT column_name, data_type, udt_name
        FROM information_schema.columns
        WHERE table_name = 'langchain_pg_embedding'
        """
    ))
    for row in result:
        print(row)

('collection_id', 'uuid', 'uuid')
('embedding', 'USER-DEFINED', 'vector')
('cmetadata', 'jsonb', 'jsonb')
('id', 'character varying', 'varchar')
('document', 'character varying', 'varchar')


In [53]:
with engine.connect() as conn:
    conn.execute(sqlalchemy.text(
        """
        CREATE INDEX IF NOT EXISTS idx_embedding_hnsw 
        ON langchain_pg_embedding 
        USING hnsw (embedding vector_cosine_ops)
        WITH (m = 16, ef_construction = 64);
        """
    ))
    conn.commit()
    print("HNSW index created")

HNSW index created


In [54]:
dummy_vector = json.dumps([0.0]*1536)

with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text(
        f"""
        EXPLAIN ANALYZE
        SELECT id, embedding <=> '{dummy_vector}'::vector(1536) AS distance
        FROM langchain_pg_embedding
        ORDER BY distance
        LIMIT 4;
        """
    ))
    for row in result:
        print(row[0])

Limit  (cost=260.28..278.09 rows=4 width=45) (actual time=0.866..0.881 rows=4.00 loops=1)
  Buffers: shared hit=478
  ->  Index Scan using idx_embedding_hnsw on langchain_pg_embedding  (cost=260.28..8227.80 rows=1790 width=45) (actual time=0.864..0.879 rows=4.00 loops=1)
        Order By: (embedding <=> '[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0

In [55]:
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text(
        "SELECT COUNT(*) FROM langchain_pg_embedding"
    ))
    for row in result:
        print(f"Total rows: {row[0]}")

Total rows: 1790


In [19]:
with engine.connect() as conn:
    conn.execute(sqlalchemy.text('SET enable_seqscan = off;'))
    result = conn.execute(sqlalchemy.text(
        f"""EXPLAIN ANALYZE
            SELECT id, embedding <=> '{dummy_vector}' :: vector(1536) AS distance
            FROM  langchain_pg_embedding
            ORDER  BY distance
            LIMIT 4;
        """
    ))

    for row in result:
        print(row[0])

Limit  (cost=260.28..278.09 rows=4 width=45) (actual time=0.411..0.425 rows=4.00 loops=1)
  Buffers: shared hit=369
  ->  Index Scan using idx_embedding_hnsw on langchain_pg_embedding  (cost=260.28..8227.80 rows=1790 width=45) (actual time=0.410..0.423 rows=4.00 loops=1)
        Order By: (embedding <=> '[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0

In [56]:
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text(
        """
            SELECT COUNT(*) FROM  langchain_pg_embedding
"""
    ))
    for row in result:
        print(f'total rows: {row[0]}')

total rows: 1790


In [57]:
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text(
        "SELECT indexname FROM pg_indexes WHERE tablename = 'langchain_pg_embedding'"
    ))
    for row in result:
        print(row)

('langchain_pg_embedding_pkey',)
('ix_cmetadata_gin',)
('idx_embedding_hnsw',)


In [22]:
# db = PGVector.from_documents(
#     embedding=embeddings,
#     documents=docs,
#     collection_name='document1',
#     connection =CONNECTION_STRING,
# )

# print(f"Successfully stored {len(docs)} chunks in PostgreSQL.")

In [ ]:
# with engine.connect() as conn:
#     conn.execute(sqlalchemy.text("TRUNCATE TABLE langchain_pg_embedding"))
#     conn.commit()
#     print("Table cleared")

# # verify
# with engine.connect() as conn:
#     result = conn.execute(sqlalchemy.text("SELECT COUNT(*) FROM langchain_pg_embedding"))
#     for row in result:
#         print(f"Rows after truncate: {row[0]}")

Table cleared
Rows after truncate: 0


In [24]:
retriever = db.as_retriever()

In [25]:
@tool(description = 'content_and artifact')
def retrieve_context(query: str):
    retrieved_doc = db.similarity_search(query, k = 3)
    serialized = "\n\n".join(f"Source: {doc.metadata} \n Content: {doc.page_content}" for doc in retrieved_doc)

    return serialized, retrieved_doc



def build_context_with_budget(raw_results, threshold, max_tokens, encoding_name = 'cl100k_base'):
    result = []
    enc = tiktoken.get_encoding(encoding_name)
    token_count = 0
    for doc, score in sorted(raw_results, key= lambda x: x[1]):
        if score > threshold:
            continue
        chunk_tokens = len(enc.encode(doc.page_content))
        if chunk_tokens + token_count >  max_tokens:
            print(f'Context budget reached at {token_count} tokens - dropping remaining chunks')
            break
        result.append(doc)
        token_count += chunk_tokens
    print(f'Context assembled {len(result)} chunks, {token_count} tokens')
    return result

In [26]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# from langchain.evaluation.qa import QAGenerateChain, QAEvalChain

# template = """You are a helpful assistant. You will be provided with a document and a question.
# Your task is to answer the question based on the content of the document. 
# here are the relevant section: {context}
# Question: {question}
# These are the past messages between you and the user, please use this history list to answer the
# question as well.
# History: {history}
# """

MAX_HISTORY_TOKENS = 3000
MAX_CONTEXT_TOKENS = 1000 # TESTING

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful ML assistant.

Use ONLY the following retrieved context to answer the question.
If the answer cannot be found in the context, you MUST:
- Set is_in_document to false
- State that the topic is not covered in the provided material
- Do NOT answer from general knowledge

Context: {context}

You must respond with the following fields:
- topic: the main ML topic the question is about
- is_in_document: true only if the answer is found in the context above
- source_chunks_used: list of direct quotes from the context you used
- answer: your answer based solely on the context provided
"""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])
# use_tools = [retrieve_context]
# agent = create_agent(model, use_tools, system_prompt= prompt) #set up template and the tools





In [27]:
# implement thread-level memory (short term memory)
store = {}
def get_section_history(session_id) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


In [28]:
def token_calculation(log, embedding_tokens, callback):
    embedding_rate = 0.1 # 0.1 USD per million input tokens embedding
    input_rate = 0.5 # 0.5 usd per million prompt token
    output_rate= 1.5 #1.5 USD per million output token
    print(f'inital question has {embedding_tokens} tokens')

    embedding_price = (embedding_tokens / 1e6)* embedding_rate
    input_cost = (callback.prompt_tokens/ 1e6) * input_rate
    output_cost = (callback.completion_tokens/1e6)*output_rate
    round_total = embedding_price + input_cost + output_cost


    log['embedding_tokens'] += embedding_tokens
    log['embedding_cost']   += embedding_price
    log['input_tokens']     += callback.prompt_tokens
    log['input_cost']       += input_cost
    log['output_tokens']    += callback.completion_tokens
    log['output_cost']      += output_cost
    log['total_tokens']     += callback.total_tokens
    log['total_cost']       += round_total

    return log
    



In [29]:
def num_tokens_from_strings(string, encoding_name):
    try:
        encoding = tiktoken.get_encoding(encoding_name)
        num_tokens = len(encoding.encode(string))
        print('num_token used for question', num_tokens)
        return num_tokens
    except KeyError:
        print('wrong encoding model')

    


In [30]:
def history_token_estimation(history: ChatMessageHistory, model: str):
    enc = tiktoken.encoding_for_model(model)
    total = 0
    for message in history.messages:
        print(f'message:{message}')
        total += len(enc.encode(message.content))
    return total




In [31]:
def daily_usage_check(user_log):
    session_budget = 0.5 # session budget set to 0.5 USD
    if user_log['total_cost'] > session_budget:
        return False

    return True
        # raise Exception("Session budget exceeded stopping the chat")

In [32]:
def pre_flight_check(messages, embed_model):
    nums_token_estimate = num_tokens_from_strings(messages, embed_model)
    if nums_token_estimate > MAX_CONTEXT_TOKENS:
        print(f"Question too long: {nums_token_estimate} tokens (max {MAX_CONTEXT_TOKENS})")
        return False

    return True


In [33]:
# structured LLM output (pydantic)
class LLMoutput(BaseModel):
    topic: str
    is_in_document: bool
    source_chunks_used: list[str]
    answer: str

class RAGResponse(BaseModel):
    answer: str
    topic: str
    is_in_document: bool
    source_title: str
    source_pages: list[int]
    confidence: str  #high, medium, low

class ChunkReference(BaseModel):
    page: int
    score: float
    snippet: str # take the first 100 cahrs for human readibilty


structured_model = model.with_structured_output(LLMoutput)


c:\Users\panlw\Desktop\ML-helper-RAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:2380: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [34]:

section_id = str(uuid7())

chain = prompt | structured_model
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_section_history,
    input_messages_key='question',
    history_messages_key='history'
)

log = {
    'embedding_tokens': 0,
    'embedding_cost': 0,
    'input_tokens': 0,
    'input_cost': 0,
    'output_tokens': 0,
    'output_cost': 0,
    'total_tokens': 0,
    'total_cost': 0
}

while True:
    print("--------------------------------------------")
    question = input("Enter your question (or 'exit' to quit): ")
    print(f'Human question:{question}')

    if question.lower() == 'exit':
        break

    if pre_flight_check(question, 'cl100k_base') is False:
        raise Exception('failed the pre-flight check')

    embedded_question = embeddings.embed_query(question)
    num_embed_tok = num_tokens_from_strings(question, "cl100k_base")

    raw_results = db.similarity_search_with_score_by_vector(embedded_question)

    # print raw scores so you can calibrate threshold

    # build chunk references 
    for doc, score in raw_results:
        print(f"Score: {score:.4f} | Page: {doc.metadata.get('page')}")
    
    chunk_reference = [
    {
        'page': doc.metadata.get("page"),
        'score': round(score, 4),
        'snippet': doc.page_content[:100]
    }
    for doc, score in raw_results if score <= 0.3
]
        

    context = build_context_with_budget(raw_results, threshold=0.3, max_tokens=MAX_CONTEXT_TOKENS)

    if not chunk_reference:
        print("No relevant chunks found above threshold")

    for doc in chunk_reference:
        print(doc)

    current_history = get_section_history(section_id)
    history_tokens = history_token_estimation(current_history, model.model_name)

    if not daily_usage_check(log):
        print('You have reached the maximum tokens, output is truncated')
        print(f'history_tokens:{history_tokens}')
        break

    print(f'history_tokens:{history_tokens}')

    with get_openai_callback() as cb:
        result = chain_with_history.invoke(
            {"question": question, "context": context},
            config={"configurable": {"session_id": section_id}}
        )
        print(f"Input tokens:  {cb.prompt_tokens}")
        print(f"Output tokens: {cb.completion_tokens}")
        print(f"Total tokens:  {cb.total_tokens}")
        print(f"Cost:          ${cb.total_cost:.5f}")

        print(f'Answer: {result.answer}')
        print(f'Topic: {result.topic}')
        print(f'In document: {result.is_in_document}')
        print(f'Sources: {chunk_reference}')

        print('------')
        token_calculation(log, num_embed_tok, cb)

        print('log = ', log)
        log['last_response'] = {
            'answer': result.answer,
            'topic': result.topic,
            'is_in_document': result.is_in_document,
            'chunk_reference': chunk_reference
        }

    # write log to file
    with open('rag_logs.jsonl', 'a') as f:
        f.write(json.dumps({
            'session_id': section_id,
            'question': question,
            'answer': result.answer,
            'topic': result.topic,
            'is_in_document': result.is_in_document,
            'sources': chunk_reference,
            'tokens': {
                'input': cb.prompt_tokens,
                'output': cb.completion_tokens,
                'total': cb.total_tokens
            },
            'cost': cb.total_cost
        }) + '\n')

--------------------------------------------
Human question:whats cross validation
num_token used for question 4
num_token used for question 4
Context assembled 0 chunks, 0 tokens
No relevant chunks found above threshold
history_tokens:0


Error in RootListenersTracer.on_chain_end callback: ValueError("Expected str, BaseMessage, list[BaseMessage], or tuple[BaseMessage]. Got topic='cross validation' is_in_document=False source_chunks_used=[] answer='The topic is not covered in the provided material.'.")


Input tokens:  195
Output tokens: 30
Total tokens:  225
Cost:          $0.00014
Answer: The topic is not covered in the provided material.
Topic: cross validation
In document: False
Sources: []
------
inital question has 4 tokens
log =  {'embedding_tokens': 4, 'embedding_cost': 4e-07, 'input_tokens': 195, 'input_cost': 9.75e-05, 'output_tokens': 30, 'output_cost': 4.5e-05, 'total_tokens': 225, 'total_cost': 0.0001429}
--------------------------------------------
Human question:exit


In [35]:
GOLDEN_DATASET = [
    {
        'question': 'What is cross-validation?',
        'expected_page': 21,  # page where answer lives
        'expected_topic': 'cross-validation'
    },
    {
        'question': 'What is overfitting?',
        'expected_page': 20,
        'expected_topic': 'overfitting'
    },
    {
        'question': 'What is k-fold cross-validation?',
        'expected_page': 21,
        'expected_topic': 'cross-validation'
    },
    {
        'question': 'What is supervised learning',
        'expected_page': 7,  # find this yourself from the book
        'expected_topic': 'supervised learning'
    },
    {
        'question': 'What is stratified k-fold?',
        'expected_page': 22,  # find this yourself
        'expected_topic': 'cross-validation'
    }
]

In [58]:
def evaluate_retrieval(golden_dataset, db, embeddings, thresholds =  0.3):
    results = []
    for item in golden_dataset:
        embedded_q = embeddings.embed_query(item['question'])
        raw_results = db.similarity_search_with_score_by_vector(embedded_q)
        retrieved_pages = [
            doc.metadata.get('page') 
            for doc, score in raw_results 
            if score <= thresholds
        ]
        
        hit = item['expected_page'] in retrieved_pages if item['expected_page'] else None
        
        results.append({
            'question': item['question'],
            'expected_page': item['expected_page'],
            'retrieved_pages': retrieved_pages,
            'hit': hit,
            'top_score': round(raw_results[0][1], 4) if raw_results else None
        })
    
    hits = [r for r in results if r['hit'] is True]
    total_scoreable = [r for r in results if r['hit'] is not None]
    recall = len(hits) / len(total_scoreable) if total_scoreable else 0
    
    print(f"\nRetrieval Recall@K: {recall:.0%} ({len(hits)}/{len(total_scoreable)})")
    print(f"Threshold: {thresholds}\n")
    
    for r in results:
        status = '✅' if r['hit'] else '❌' if r['hit'] is False else '⚠️'
        print(f"{status} Q: {r['question'][:50]}")
        print(f"   Expected page: {r['expected_page']} | Retrieved: {r['retrieved_pages']} | Top score: {r['top_score']}")
    
    return results

In [64]:
result = evaluate_retrieval(GOLDEN_DATASET, db, embeddings, thresholds = 0.3)
print(result)


Retrieval Recall@K: 100% (5/5)
Threshold: 0.3

✅ Q: What is cross-validation?
   Expected page: 21 | Retrieved: [13, 21, 20, 28] | Top score: 0.1112
✅ Q: What is overfitting?
   Expected page: 20 | Retrieved: [19, 19, 20, 102] | Top score: 0.1287
✅ Q: What is k-fold cross-validation?
   Expected page: 21 | Retrieved: [21, 22, 25, 20] | Top score: 0.1243
✅ Q: What is supervised learning
   Expected page: 7 | Retrieved: [6, 7, 6, 8] | Top score: 0.152
✅ Q: What is stratified k-fold?
   Expected page: 22 | Retrieved: [22, 24, 25, 26] | Top score: 0.1311
[{'question': 'What is cross-validation?', 'expected_page': 21, 'retrieved_pages': [13, 21, 20, 28], 'hit': True, 'top_score': 0.1112}, {'question': 'What is overfitting?', 'expected_page': 20, 'retrieved_pages': [19, 19, 20, 102], 'hit': True, 'top_score': 0.1287}, {'question': 'What is k-fold cross-validation?', 'expected_page': 21, 'retrieved_pages': [21, 22, 25, 20], 'hit': True, 'top_score': 0.1243}, {'question': 'What is supervised 

In [65]:
result = evaluate_retrieval(GOLDEN_DATASET, db_v2, embeddings_v2, thresholds = 0.45)
print(result)


Retrieval Recall@K: 60% (3/5)
Threshold: 0.45

❌ Q: What is cross-validation?
   Expected page: 21 | Retrieved: [20, 13, 28, 25] | Top score: 0.3495
✅ Q: What is overfitting?
   Expected page: 20 | Retrieved: [19, 19, 20] | Top score: 0.3703
✅ Q: What is k-fold cross-validation?
   Expected page: 21 | Retrieved: [22, 25, 22, 21] | Top score: 0.3099
❌ Q: What is supervised learning
   Expected page: 7 | Retrieved: [6] | Top score: 0.3739
✅ Q: What is stratified k-fold?
   Expected page: 22 | Retrieved: [23, 22, 22, 26] | Top score: 0.3571
[{'question': 'What is cross-validation?', 'expected_page': 21, 'retrieved_pages': [20, 13, 28, 25], 'hit': False, 'top_score': 0.3495}, {'question': 'What is overfitting?', 'expected_page': 20, 'retrieved_pages': [19, 19, 20], 'hit': True, 'top_score': 0.3703}, {'question': 'What is k-fold cross-validation?', 'expected_page': 21, 'retrieved_pages': [22, 25, 22, 21], 'hit': True, 'top_score': 0.3099}, {'question': 'What is supervised learning', 'expec

In [39]:
question = "what is cross-validation"
embedded_q = embeddings_v2.embed_query(question)
results = db_v2.similarity_search_with_score_by_vector(embedded_q)

for doc, score in results:
    print(f"Score: {score:.4f} | Page: {doc.metadata.get('page')}")
    print(f"Content: {doc.page_content[:150]}")
    print("---")

In [61]:
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text(
        """
        SELECT e.cmetadata, e.document
        FROM langchain_pg_embedding e
        JOIN langchain_pg_collection c ON e.collection_id = c.uuid
        WHERE c.name = 'ml_docs_v2'
        AND e.cmetadata->>'page' = '20'
        """
    ))
    for row in result:
        print(row)

({'page': 20, 'title': 'Approaching (Almost) Any Machine Learning Problem', 'author': 'Abhishek Thakur', 'source': 'documents\\AAAMLP.pdf', 'creator':  ... (44 characters truncated) ... , 'producer': 'macOS Version 11.4 (Build 20F71) Quartz PDFContext', 'page_label': '21', 'total_pages': 300, 'creationdate': "D:20210608080427Z00'00'"}, 'or may not apply to other datasets. However, there are a few types of cross-validation techniques which are the most popular and widely used.   These include:  • k-fold cross-validation • stratified k-fold cross-validation')
({'page': 20, 'title': 'Approaching (Almost) Any Machine Learning Problem', 'author': 'Abhishek Thakur', 'source': 'documents\\AAAMLP.pdf', 'creator':  ... (44 characters truncated) ... , 'producer': 'macOS Version 11.4 (Build 20F71) Quartz PDFContext', 'page_label': '21', 'total_pages': 300, 'creationdate': "D:20210608080427Z00'00'"}, 'Approaching (Almost) Any Machine Learning Problem – Abhishek Thakur \n 21 \nOccam’s razor in simp

In [62]:
# Print the full retrieved chunks to see if it actually found the explanation
for i, (doc, score) in enumerate(results):
    print(f"Rank {i+1} | Score: {score:.4f} | Page: {doc.metadata.get('page')}")
    print(doc.page_content)
    print("="*50)

In [63]:
with engine.connect() as conn:
    # check row counts per collection
    result = conn.execute(sqlalchemy.text(
        """
        SELECT c.name, COUNT(e.id) as chunk_count
        FROM langchain_pg_collection c
        LEFT JOIN langchain_pg_embedding e ON e.collection_id = c.uuid
        GROUP BY c.name
        """
    ))
    for row in result:
        print(f"Collection: {row[0]} | Chunks: {row[1]}")

Collection: document1 | Chunks: 0
Collection: ml_docs_v2 | Chunks: 895
Collection: ML knowledge | Chunks: 895
Collection: AAAMLP.pdf | Chunks: 0
